## setup 

we'll generate some dummy training data first for this simulated run

In [ ]:
from loss_landscapes.datagen.func2d import generate_artificial_data
from loss_landscapes.datagen.io import write_data

data, targets = generate_artificial_data(n_datapoints = 100_000)
write_data("ds_func2d.h5", data, targets)

## training the model

we can only visualize the loss landscape of a trained model, otherwise it's going to be a generic smooth trough shape. so, a very simple network is trained to fit this 2D -> 1D function data.

In [ ]:
import torch

from loss_landscapes.train.setup import get_device
from loss_landscapes.datagen.io import load_data
from loss_landscapes.networks.simple import SimpleNet
from loss_landscapes.train.setup import TrainingConfig, DataSplit
from loss_landscapes.train.loop import training_loop

device = get_device()
data, targets = load_data("ds_func2d.h5")
model = SimpleNet().to(device)
optim = torch.optim.Adam(params = model.parameters(), lr=1e-3)
lossf = lambda pred, targ: torch.sqrt(torch.mean((pred - targ)**2))

cfg = TrainingConfig(
    n_epochs = 10,
    batch_size = 256,
    train_split = 0.8,
)
dsp = DataSplit(
    n_total = len(data),
    n_train = int(cfg.train_split * len(data))
)

data = torch.as_tensor(data, dtype=torch.float32).to(device)
targets = torch.as_tensor(targets, dtype=torch.float32)[:, None].to(device)

_ = training_loop(
    cfg, dsp,
    data, targets,
    device, model, optim, lossf
)

## Evaluating the Loss Landscape

#### "naive normalization" of random parameter direction

in order to evaluate the loss landscape of the model around the trained weights, we need two random directions in model parameter space. generating them is fairly straightforward; just step through all the models layers and create a tensor of random numbers with euqal shape as the layers weights. However, these randomly generated numbers are likely to be totally disproportionate to the models actual weights! so it would be really hard to know, how much we are disturbing the parameters with these random vectors. 

The first instinctive choice might be to normalize this random parmeter vector to some scalar value, ideally the same as the norm of the entire models weight "vector" (global normalization, or called "naive" normalization here). this does kind of work, and especially for a small dummy model like the one here, it is sufficient. (biases and buffers of dim = 1 are skipped).

In [ ]:
from loss_landscapes.inference.evaluate import generate_random_dir_naive
from loss_landscapes.inference.evaluate import evaluate_grid

trained_params = {k: v.detach().clone() for k, v in model.named_parameters()}
dir1 = generate_random_dir_naive(trained_params)
dir2 = generate_random_dir_naive(trained_params)

x, y, z = evaluate_grid(
    data, targets,
    model, trained_params, device, lossf,
    dir1, dir2, reach = 1.0, npoints = 51,
)
z = z.cpu().numpy()

In [ ]:
import numpy as np
import plotly.graph_objects as go 
from loss_landscapes.plotly.templates import use
use("mine_dark")

go.Figure(
    data = go.Surface(
        x = x,
        y = y,
        z = np.log(z),
    ),
    layout = dict(
        margin = dict(l=20, r=20, t=20, b=20),
        height = 400,
    ),
).show()

#### proper "filter normalization"

especially for larger and deeper models, this naive global normalization will distort the loss landscape. Inside a model, features can technically be scaled independently, as the model can adjust its weights to different scales. Especially for ResNets with batchnorm after each block, any absolute scale for the features is lost, as batch norm always rescales them anyways. Therefore, a uniformely normalized direction might emphasize/explore different filters with different strength - depending on their "internal scale"

to solve this problem [https://arxiv.org/abs/1712.09913] Li et al successfully use per-filter normalization of the random directions to match the trained filters norms. This means, instead of normalizing the entire parameter vector, each kernel weight matrix is normalized individually for example, to respect it scale.

In [ ]:
from loss_landscapes.inference.evaluate import generate_random_dir_filternorm
from loss_landscapes.inference.evaluate import evaluate_grid


trained_params = {k: v.detach().clone() for k, v in model.named_parameters()}
dir1 = generate_random_dir_filternorm(trained_params)
dir2 = generate_random_dir_filternorm(trained_params)

x, y, z = evaluate_grid(
    data, targets,
    model, trained_params, device, lossf,
    dir1, dir2, reach = 1.0, npoints = 51,
)
z = z.cpu().numpy()

In [ ]:
import numpy as np
import plotly.graph_objects as go 
from loss_landscapes.plotly.templates import use
use("mine_dark")

go.Figure(
    data = go.Surface(
        x = x,
        y = y,
        z = np.log(z),
    ),
    layout = dict(
        margin = dict(l=20, r=20, t=20, b=20),
        height = 400,
    ),
).show()

for this small model, the difference in normalization is not really apparent and both surfaces look very similar. in the next notebook, we'll examine a decently sized resnet to see the influence of normalization better